In [1]:
import numpy as np
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt


In [9]:
import idna #implementacion de tecnicas de codificacion de strings (codificacion y decodificacion de caracterers unicode)

# def utf8_to_punycode(text: str) -> str:
#     """Encodes a UTF-8 string to its Punycode representation."""
#     return idna.encode(text).decode('ascii')

def punyencode(text: str) -> str:
    """Encodes a UTF-8 string to its Punycode representation, handling spaces by encoding each word separately."""
    
    return " ".join([idna.encode(word).decode('ascii') for word in text.split()])
    
def punydecode(punycode: str) -> str:
    """Decodes a Punycode string back to UTF-8."""
    #return idna.decode(punycode)
    return " ".join([idna.decode(word) for word in punycode.split()])

def process_name(name):
    name = name.lower()
    for n in name.split():
        if len(n) < 2:
            return ''
    try:
        return punyencode(name)
    except:
        #print(f'Cant convert {name}')
        return ''

dataset = open("city_names_full.txt", 'r', encoding='utf-8').read().split('\n')
with open('city_names_puny.txt', 'w', encoding='utf-8') as f:
    for n in dataset:
        name = process_name(n)
        if name != '':
            f.write(name + '\n')
dataset = open('city_names_puny.txt', 'r', encoding='utf-8').read().split('\n')
puny = [x for x in dataset if 'xn--' in x] #filtrar todos los que tienen caracteres raros
nopuny = [x for x in dataset if 'xn--' not in x]
np.random.seed(42)
dataset = [x.item() for x in np.random.choice(nopuny, 100000, replace=False)]

In [11]:
#construyendo dataset para makemore con MLPS

In [14]:
print(len(dataset))
charset = ['*'] + sorted(list(set([y for x in dataset for y in x])))
ctoi = {c:i for i, c in enumerate(charset)}
itoc = {i:c for i, c in enumerate(charset)}
charset_size = len(charset)


100000


In [17]:
context_size = 3
X, Y = [], []

for d in dataset[:1]:
    print(d)
    example = list(d) + ['*']
    context = [0] * context_size
    for c in example:
        print(''.join([itoc[x] for x in context])+ '---> + c')
        X.append(context)
        Y.append(ctoi[c])
        context = context[1:] + [ctoi[c]]

kotamangalam
***---> + c
**k---> + c
*ko---> + c
kot---> + c
ota---> + c
tam---> + c
ama---> + c
man---> + c
ang---> + c
nga---> + c
gal---> + c
ala---> + c
lam---> + c


In [19]:
X, Y

([[0, 0, 0],
  [0, 0, 24],
  [0, 24, 28],
  [24, 28, 33],
  [28, 33, 14],
  [33, 14, 26],
  [14, 26, 14],
  [26, 14, 27],
  [14, 27, 20],
  [27, 20, 14],
  [20, 14, 25],
  [14, 25, 14],
  [25, 14, 26]],
 [24, 28, 33, 14, 26, 14, 27, 20, 14, 25, 14, 26, 0])

In [20]:
#Red neuronal

In [22]:
g = torch.Generator().manual_seed(42)
emb_size = 2
C = torch.randn(charset_size, emb_size, generator=g)#genera una matriz de 40*2
C

tensor([[ 1.9269,  1.4873],
        [ 0.9007, -2.1055],
        [ 0.6784, -1.2345],
        [-0.0431, -1.6047],
        [-0.7521,  1.6487],
        [-0.3925, -1.4036],
        [-0.7279, -0.5594],
        [-0.7688,  0.7624],
        [ 1.6423, -0.1596],
        [-0.4974,  0.4396],
        [-0.7581,  1.0783],
        [ 0.8008,  1.6806],
        [ 1.2791,  1.2964],
        [ 0.6105,  1.3347],
        [-0.2316,  0.0418],
        [-0.2516,  0.8599],
        [-1.3847, -0.8712],
        [-0.2234,  1.7174],
        [ 0.3189, -0.4245],
        [ 0.3057, -0.7746],
        [-1.5576,  0.9956],
        [-0.8798, -0.6011],
        [-1.2742,  2.1228],
        [-1.2347, -0.4879],
        [-0.9138, -0.6581],
        [ 0.0780,  0.5258],
        [-0.4880,  1.1914],
        [-0.8140, -0.7360],
        [-1.4032,  0.0360],
        [-0.0635,  0.6756],
        [-0.0978,  1.8446],
        [-1.1845,  1.3835],
        [ 1.4451,  0.8564],
        [ 2.2181,  0.5232],
        [ 0.3466, -0.1973],
        [-1.0546,  1

In [24]:
#onehot es un vector que dependiendo de el numero que se ingrese va a mostrar 1 en el indice del caracter que se esta codeando (?)
#F.one_hot(torch.tensor(0), num_classes = charset_size).float()
F.one_hot(torch.tensor(0), num_classes = charset_size).float() @ C

tensor([1.9269, 1.4873])

In [ ]:
F.one_hot(torch.tensor(X), num_classes = charset_size).float() @ C #devuelve una matriz con el embedding de cada uno de los char de X(pesos)

tensor([[[ 1.9269,  1.4873],
         [ 1.9269,  1.4873],
         [ 1.9269,  1.4873]],

        [[ 1.9269,  1.4873],
         [ 1.9269,  1.4873],
         [-0.9138, -0.6581]],

        [[ 1.9269,  1.4873],
         [-0.9138, -0.6581],
         [-1.4032,  0.0360]],

        [[-0.9138, -0.6581],
         [-1.4032,  0.0360],
         [ 2.2181,  0.5232]],

        [[-1.4032,  0.0360],
         [ 2.2181,  0.5232],
         [-0.2316,  0.0418]],

        [[ 2.2181,  0.5232],
         [-0.2316,  0.0418],
         [-0.4880,  1.1914]],

        [[-0.2316,  0.0418],
         [-0.4880,  1.1914],
         [-0.2316,  0.0418]],

        [[-0.4880,  1.1914],
         [-0.2316,  0.0418],
         [-0.8140, -0.7360]],

        [[-0.2316,  0.0418],
         [-0.8140, -0.7360],
         [-1.5576,  0.9956]],

        [[-0.8140, -0.7360],
         [-1.5576,  0.9956],
         [-0.2316,  0.0418]],

        [[-1.5576,  0.9956],
         [-0.2316,  0.0418],
         [ 0.0780,  0.5258]],

        [[-0.2316,  0

In [26]:
# (13, 3 ,40) @ (40, 2) --> (13, 3, 2)

In [27]:
C[[X]] #esto es el equivalente a la multiplicacion de matrices que devuelve la matriz de embeddings arriba

# Importante!
# • El tamaño contexto de nuestro modelo (cuántos caracteres se utilizan para predecir la distribución de probabilidad del siguiente) determina muchas cosas y es conveniente
# tener claro cual es este número. En nuestro ejemplo, el tamaño del contexto es 3. Es decir, utilizamos 3 caracteres para predecir el siguiente.
# • La matriz C puede interpretarse como la primer capa de perceptronoes.
# • C tiene 2 perceptrones de 40 entradas, es decir tiene 40 entradas y 2 salidas, lo que permite codificar 1 caracter en 2 dimensiones
# • Nuestro modelo no recibe como entrada un one hot de 40 dimensiones, recibe el índice de 3 caracteres.
# • Codificar 3 caracteres requiere que multiplicar (3, 40)@(40, 2) (3, 2) es decir, primero tenemos que calcular el one hot de los 3 caracteres y despues
# multiplicar los one hot por C.
# • Durante el entrenamiento vamos a querer mandarle "batches" de 3 caracteres. Es decir, vamos a queresr enviar varios grupos de 3 caracteres. Si el tamaño del grupo es
# de 13, como vimos en el ejemplo, podemos interpretar que capa C esta recibiendo 13 grupos de 3 caracteres encodeados en one hot cuya dimensión seria (13,
# 3, 40) y lo va a multiplicar por (40, 2), es decir (13, 3, 40)@(40, 2) (13, 3, 2). La salida se puede interpretar como 13 grupos de 3 caracteres encodeados en 2
# dimensiones (las dimensiones del embedding).
# • Se puede lograr el mismo efecto indexando C mediante el uso las ténicas avanzadas de indexing de PyTorch, en lugar de hacer operaciones con matrices (guiño, guiño).
# • En general las dimensiones del modelo van a ser de
# o Entrada: (batch_size, context_size, (charsetLSize)J0, charset_size será necesario si usamos one hot encondig, se puede obviar si usamos indexing.
# o Primer capa (Matriz de embeddings): (charset_size, es decir, una red de emb_size perceptrones (emb_size salidas), cada uno con charset_size
# entradas.
# o Salida: (batch_size, context_size,emb_size)
# o (batch_size, context_size, (charset_size)) @ (charset_size, emb_size) (batch_size, context_size,emb_size)
# • Es necesario tener presente el tamaño de la salida por que de eso depende el tamaño de la entrada de la siquiente capa.

C:\Users\TheCague\AppData\Local\Temp\ipykernel_20596\934146368.py:1: UserWarning: Using a non-tuple sequence for multidimensional indexing is deprecated and will be changed in pytorch 2.9; use x[tuple(seq)] instead of x[seq]. In pytorch 2.9 this will be interpreted as tensor index, x[torch.tensor(seq)], which will result either in an error or a different result (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\pytorch\torch\csrc\autograd\python_variable_indexing.cpp:353.)
  C[[X]] #esto es el equivalente a la multiplicacion de matrices que devuelve la matriz de embeddings arriba


tensor([[[ 1.9269,  1.4873],
         [ 1.9269,  1.4873],
         [ 1.9269,  1.4873]],

        [[ 1.9269,  1.4873],
         [ 1.9269,  1.4873],
         [-0.9138, -0.6581]],

        [[ 1.9269,  1.4873],
         [-0.9138, -0.6581],
         [-1.4032,  0.0360]],

        [[-0.9138, -0.6581],
         [-1.4032,  0.0360],
         [ 2.2181,  0.5232]],

        [[-1.4032,  0.0360],
         [ 2.2181,  0.5232],
         [-0.2316,  0.0418]],

        [[ 2.2181,  0.5232],
         [-0.2316,  0.0418],
         [-0.4880,  1.1914]],

        [[-0.2316,  0.0418],
         [-0.4880,  1.1914],
         [-0.2316,  0.0418]],

        [[-0.4880,  1.1914],
         [-0.2316,  0.0418],
         [-0.8140, -0.7360]],

        [[-0.2316,  0.0418],
         [-0.8140, -0.7360],
         [-1.5576,  0.9956]],

        [[-0.8140, -0.7360],
         [-1.5576,  0.9956],
         [-0.2316,  0.0418]],

        [[-1.5576,  0.9956],
         [-0.2316,  0.0418],
         [ 0.0780,  0.5258]],

        [[-0.2316,  0

In [29]:
#Capa oculta (capa de perceptrones con una funcion de activacion) (capa de neuronas)
emb = C[[X]] #embeddings del dataset (pesos)

C:\Users\TheCague\AppData\Local\Temp\ipykernel_20596\501557912.py:2: UserWarning: Using a non-tuple sequence for multidimensional indexing is deprecated and will be changed in pytorch 2.9; use x[tuple(seq)] instead of x[seq]. In pytorch 2.9 this will be interpreted as tensor index, x[torch.tensor(seq)], which will result either in an error or a different result (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\pytorch\torch\csrc\autograd\python_variable_indexing.cpp:353.)
  emb = C[[X]] #embeddings del dataset (pesos)


In [ ]:

#definicion de la capa oculta
n_hidden = 128 #128 neuronas, cada una con sus entradas y salidas
W1 = torch.randn(emb_size * context_size,  n_hidden) #el contexto esta enbebido en 2 dimensiones y son 3 caracteres en el contexto, cantidad de perceptrones
b1 = torch.randn(n_hidden) #bias

In [ ]:

emb [0, 0], emb[0,1], emb[0,2]
#forward pass
F.tanh(emb[0].view(-1, 6) @ W1 + b1 )#salida de la capa C por la capa de perceptrones, aplicandole la funcion de activacion(tanh)


tensor([[ 0.9942, -0.9998,  0.9996,  0.9863,  0.8782, -0.9999, -0.9995,  0.9968,
          1.0000, -1.0000, -1.0000, -0.5705, -0.9146, -0.9999, -0.1389, -0.8972,
          0.9206, -1.0000, -1.0000, -1.0000,  0.9615, -0.9982, -0.9919,  1.0000,
         -0.4820,  1.0000,  0.9971, -0.9990,  0.9305, -1.0000,  0.9958, -0.9862,
          0.9957, -1.0000, -0.7532, -1.0000, -0.6996,  0.9870,  1.0000,  0.9792,
          0.9964,  1.0000,  0.9329, -1.0000,  1.0000,  0.9993,  0.9985,  0.9908,
         -0.9999, -0.9890,  0.9942, -0.9992,  0.1675, -1.0000,  1.0000, -1.0000,
          0.9980,  0.9975,  0.1546,  0.9417,  0.5282,  1.0000,  0.9244,  1.0000,
          0.9988,  0.9678,  0.9946, -0.9999, -0.2660, -0.9919,  0.4484,  0.9992,
          0.9982, -0.2358,  0.5520,  0.8851,  0.9982, -0.9999,  0.9992,  1.0000,
         -1.0000,  0.9985,  0.8533,  0.9999, -0.9920,  1.0000, -0.9950,  0.1094,
          1.0000, -0.9996,  0.9235, -1.0000,  0.9955,  0.9993, -0.1495,  1.0000,
          0.5286,  0.9981,  

In [ ]:
# (1,6) @ (6, 128) -> si le se suma 'algo de 128' se conoce como broadcasting rules
# la suma de una listas de listas y una lista, siempre y cuando:
# compartan las mismas dimensiones, o que una de las dimensiones sea 1 o una de las dimensiones no existe
# cada tensor tiene por lo menos una dimension


In [46]:
torch.tensor([[1,2,3,4], [2,3,4,5]]) + torch.tensor([1,2,3,4])
torch.tensor([[1,2,3,4], [2,3,4,5]]).shape, torch.tensor([1,2,3,4]).shape
# 2, 4(en este caso los 4 tenian que ser iguales)
# X, 4

(torch.Size([2, 4]), torch.Size([4]))

In [47]:
#capa output

W2 = torch.randn(n_hidden, charset_size)
b2 = torch.randn(charset_size)


In [49]:
act = F.tanh(emb[0].view(-1, 6) @ W1 + b1 )

In [57]:
#(13,128) @ (128, 40) --> (13, 40)
logits = act @ W2 + b2
counts = logits.exp()
probs = counts/counts.sum(axis=1, keepdims = True)

#e ** log(x)

In [58]:
probs

tensor([[7.0589e-04, 7.8386e-11, 3.4077e-07, 5.8845e-06, 2.7476e-17, 2.0323e-08,
         3.7515e-08, 1.5530e-06, 2.2893e-05, 7.3526e-11, 1.7614e-09, 9.9884e-01,
         6.8282e-17, 1.1863e-07, 1.3276e-09, 4.4100e-14, 6.2306e-09, 3.8204e-11,
         9.1890e-07, 1.8378e-08, 2.8820e-10, 2.6953e-15, 4.5778e-07, 2.8351e-09,
         1.1653e-09, 4.7021e-11, 7.9032e-07, 2.0656e-11, 1.5642e-11, 2.7288e-07,
         1.8128e-11, 5.5642e-12, 6.6454e-05, 2.0985e-07, 2.2958e-07, 1.0645e-13,
         3.8440e-10, 2.1540e-07, 3.4310e-04, 1.4839e-05]])